# TP · Identidad y emoción de actores argentinos — Grupo 6 (versión final)
**IAyAA II · Licenciatura en Ciencia de Datos · UCA Rosario · 2026**

Martín Masuelli · Francisco Miles · Tomás Lizazu · Bautista Verna · Juan Diego Ancarani · Manuel Lafosse

Dos tareas sobre los recortes de rostro del curso: **identidad** (20 actores) y **emoción** (alegría, enojo, neutral, sorpresa, tristeza).

Este notebook contiene solo lo que se entrega: **la CNN propia más efectiva** (entrenada desde cero) y **ResNet50** con transfer learning. Las versiones anteriores de la CNN quedan documentadas en el **Anexo A** (plegado). El código completo de esas versiones y de la búsqueda con KerasTuner está en `notebooks/TP2_Grupo6.ipynb` del repositorio; cada experimento en `docs/EXPERIMENTOS_CNN.md` y cada decisión en `docs/BITACORA.md`.

| # | Sección |
|---|---|
| 1 | Preparación del entorno y datos (Drive del curso en solo lectura) |
| 2 | Datos: registro, video de origen, distribución y ejemplos |
| 3 | Unificación de videos, partición **por video** y control de fugas |
| 4 | Baselines (trivial, Random Forest, MLP) |
| 5 | Acta de límites (NB02) |
| 6 | Pipeline `tf.data`, aumentación y herramientas de entrenamiento |
| 7 | **CNN propia final** (una red por tarea) y revancha de la permutación |
| 8 | ResNet50 (transfer learning) |
| 9 | Comparación en validación y recall por actor |
| 10 | Evaluación final en test (una sola vez): matrices de confusión y peores errores |
| 11 | Interpretación: ¿el modelo mira la cara? (oclusión) |
| 12 | Predicción sobre fotos nuevas |
| 13 | Conclusiones y limitaciones |
| A | Anexo (plegado): versiones anteriores de la CNN |

**Requisitos:** GPU T4 y, en la raíz de "Mi unidad", un acceso directo a `CARPETA COMPARTIDA PARA TODOS` (y, si la cuenta no ve las imágenes ahí, otro a `Imagenes_Finales`). Tiempo total ≈ 55 min.

## 1. Preparación del entorno
Toda constante que afecta un resultado está en la celda 1.1. Los hiperparámetros de la CNN son los que eligió **KerasTuner** (NB09) en la búsqueda del notebook completo (experimento E16); acá se usan fijos para no repetir una búsqueda de 1,5 h.

In [ ]:
# 1.1 Librerías, configuración y semillas
import os, re, glob, time, random, shutil, hashlib, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from IPython.display import display

import tensorflow as tf
import keras
from keras import layers
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED = 42
EMOCIONES = ("alegria", "enojo", "neutral", "sorpresa", "tristeza")
TAREAS = ("actor", "emocion")
FRACCIONES = {"train": 0.70, "val": 0.15, "test": 0.15}
CSV_MASTER, CARPETA_IMAGENES = "etiquetas_master_final.csv", "Imagenes_Finales"
IMG_SIZE = 224             # recortes del curso = entrada nativa de ResNet50
IMG_SIZE_BASELINE = 64     # baselines clásicos sobre píxeles en gris

# CNN propia final: hiperparámetros elegidos por KerasTuner por accuracy de validación (E16, D-018/D-019)
IMG_SIZE_CNN, BATCH_CNN = 112, 64
HP_CNN = {"base": 48, "dropout": 0.4, "l2": 0.0, "lr": 1.4e-3, "rotacion": 0.04, "zoom": 0.10, "color": 0.10}
EPOCAS_MAX_CNN, PACIENCIA_CNN = 150, 15

# ResNet50: fase 1 cabeza (lr 1e-3), fase 2 descongelando conv4 + conv5 (lr 1e-5), elegido en validación (E15)
BATCH_RESNET = 32
RESNET_PRIMERA_DESCONGELADA = "conv4_block1_1_conv"

# Partición: dos videos del mismo actor con alguna pareja de fotos más parecida que esto se tratan como UN video (3.1, D-022)
UMBRAL_MISMO_VIDEO = 0.82

# Identidades que el CSV del curso escribe distinto
MAPA_ACTORES = {
    "tini": "tini_stoessel", "georginabarbarossa": "georgina_barbarossa",
    "oscarmartinez": "oscar_martinez", "juliochavez": "julio_chavez",
    "pachupena": "pachu_pena", "rodrigodelaserna": "rodrigo_de_la_serna",
    "francella": "guillermo_francella", "darin": "ricardo_darin", "lali": "lali_esposito",
    "milei": "javier_milei", "barassi": "dario_barassi", "momo_benavides": "momo",
    "geronimo_benavides": "momo", "flavio_azzaro": "azzaro",
    "franco_colapinto": "colapinto", "lionel_scaloni": "scaloni",
}


def fijar_semillas(seed=SEED):
    # Fija todas las fuentes de azar; se llama al inicio y antes de construir cada modelo.
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed)
    keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


@keras.saving.register_keras_serializable(package="tp_g6")
class PreprocesoResNet50(layers.Layer):
    # Cada base preentrenada exige SU preprocesamiento (NB13): RGB->BGR y resta de la media de ImageNet.
    # Se define acá (y no en la sección 8) para poder cargar la ResNet50 guardada sin volver a entrenar (12.2).
    def call(self, x):
        return keras.applications.resnet50.preprocess_input(x)


fijar_semillas()
sns.set_theme(style="whitegrid")
print(f"TensorFlow {tf.__version__} · Keras {keras.__version__}")
print("GPU:", tf.config.list_physical_devices("GPU") or "NO — activar GPU T4 en Entorno de ejecución")

In [ ]:
# 1.2 Drive del curso en SOLO LECTURA + copia local de trabajo
from google.colab import drive
drive.mount("/content/drive", readonly=True)   # readonly: es imposible modificar el Drive del curso

candidatas = (glob.glob("/content/drive/MyDrive/CARPETA COMPARTIDA PARA TODOS")
              + glob.glob("/content/drive/MyDrive/*/CARPETA COMPARTIDA PARA TODOS"))
assert candidatas, "Falta el acceso directo a 'CARPETA COMPARTIDA PARA TODOS' en Mi unidad."
CARPETA_CURSO = Path(candidatas[0])
# Si una cuenta no ve las imágenes dentro del dataset del curso (permisos), sirve un acceso directo a la carpeta de imágenes.
origen_imgs = next((p for p in (CARPETA_CURSO / "DATASET_FINAL" / CARPETA_IMAGENES,
                                Path("/content/drive/MyDrive") / CARPETA_IMAGENES,
                                Path("/content/drive/MyDrive/DATASET_FINAL") / CARPETA_IMAGENES) if p.exists()), None)
assert origen_imgs, f"Esta cuenta no ve '{CARPETA_IMAGENES}': pedir acceso de lector o agregar un acceso directo en Mi unidad."
DATASET = Path("/content/datos/DATASET_FINAL")
if not (DATASET / CARPETA_IMAGENES).exists():      # copia local: más rápida y aísla el original
    shutil.rmtree(DATASET, ignore_errors=True)       # descarta una copia incompleta (disco de Colab, no el Drive)
    shutil.copytree(CARPETA_CURSO / "DATASET_FINAL", DATASET, ignore=shutil.ignore_patterns("Metricas_Dataset", CARPETA_IMAGENES))
    shutil.copytree(origen_imgs, DATASET / CARPETA_IMAGENES)
print("Imágenes tomadas de:", origen_imgs)
print("Copia de trabajo:", DATASET, "→", len(os.listdir(DATASET / CARPETA_IMAGENES)), "imágenes")

## 2. Datos
El CSV maestro del curso tiene tres problemas que el código resuelve: nombres con tildes y la `ñ` codificada como `#U00f1` (se cruzan por una *clave* normalizada), identidades escritas distinto (`tini` / `tini_stoessel`, se unifican con `MAPA_ACTORES`) y, sobre todo, **no tiene el video de origen**, indispensable para partir sin fugas. El video se recupera de los registros que subió cada grupo (solo lectura) y, si una imagen no figura, del patrón de su nombre.

In [ ]:
# 2.1 Normalización, video de origen y registro (una fila por imagen utilizable)
def clave_archivo(nombre):
    # 'PachuPe#U00f1a_Alegr#U00eda.jpg' y 'PachuPeña_Alegría.jpg' -> 'pachupena_alegria.jpg'
    s = re.sub(r"#U([0-9a-fA-F]{4})", lambda m: chr(int(m.group(1), 16)), str(nombre))
    s = unicodedata.normalize("NFKD", s)
    return "".join(c for c in s if not unicodedata.combining(c)).lower().strip()


def normalizar_etiqueta(texto):
    s = re.sub(r"[\s\-_]+", "_", clave_archivo(texto))
    return re.sub(r"[^a-z0-9_]", "", s).strip("_")


def normalizar_actor(nombre):
    actor = normalizar_etiqueta(nombre)
    return MAPA_ACTORES.get(actor, actor)


PATRONES_VIDEO = (
    re.compile(r"^.+?__(?P<video>.+?)__f\d+\.jpe?g$"),          # G1/G10: actor__video__f0000123.jpg
    re.compile(r"^(?P<video>.+?)_frame_\d+\.jpe?g$"),           # G4: Cande1_frame_000010.jpg
    re.compile(r"^(?P<video>.+?)(?:_p\d+)?_\d+\.jpe?g$"),       # G7: momo_video_1_p2_0003.jpg
)


def video_desde_nombre(clave):
    for patron in PATRONES_VIDEO:
        m = patron.match(clave)
        if m:
            return m.group("video")
    return None


def leer_registros_grupos(carpeta_curso):
    # Mapa clave -> video a partir del registro por imagen de cada grupo (solo lectura).
    fuentes, cols = [], ["archivo", "actor", "emocion", "video", "escena"]
    for g in ("G3", "G7", "G8"):
        p = carpeta_curso / g / "etiquetas_master.txt"
        if p.exists():
            fuentes.append(pd.read_csv(p, sep=";", header=None, names=cols)[["archivo", "video"]])
    especiales = {"G1": ("registro_grupo.csv", "archivo", "video", {}),
                  "G4": ("labels_224_final_grupo4.csv", "imagen", "video", {}),
                  "G6": ("DATASET_FINAL.csv", "imagen", "video_origen", {"encoding": "utf-8-sig"})}
    for g, (nombre, c_arch, c_vid, kw) in especiales.items():
        p = carpeta_curso / g / nombre
        if p.exists():
            d = pd.read_csv(p, **kw).dropna(subset=[c_arch, c_vid])
            fuentes.append(pd.DataFrame({"archivo": d[c_arch], "video": d[c_vid].astype(str).str.slice(0, 24)}))
    if not fuentes:
        return {}
    todo = pd.concat(fuentes, ignore_index=True)
    todo["video"] = todo["video"].astype(str).str.replace(r"\.mp4$", "", regex=True)
    todo["clave"] = todo["archivo"].map(clave_archivo)
    return dict(todo.drop_duplicates("clave").set_index("clave")["video"])


def asignar_video(claves, registro):
    videos, origenes = [], []
    for c in claves:
        if c in registro:
            videos.append(registro[c]); origenes.append("registro del grupo")
        elif (v := video_desde_nombre(c)) is not None:
            videos.append(v); origenes.append("nombre de archivo")
        else:   # último recurso conservador: agrupa de más, nunca genera fugas
            videos.append(re.sub(r"\d+", "", Path(c).stem).strip("_")); origenes.append("prefijo (conservador)")
    return videos, origenes


def huella_dataset(df):
    # Hash de qué imágenes hay y con qué etiquetas: identifica la VERSIÓN del dataset.
    contenido = "\n".join(sorted(df["clave"] + ";" + df["actor"] + ";" + df["emocion"]))
    return hashlib.sha256(contenido.encode()).hexdigest()[:12]


def construir_registro(ruta_dataset, carpeta_curso):
    df = pd.read_csv(ruta_dataset / CSV_MASTER, sep=None, engine="python", encoding="utf-8-sig")
    df.columns = [c.strip().lower() for c in df.columns]
    fisicas = {clave_archivo(p.name): p for p in (ruta_dataset / CARPETA_IMAGENES).iterdir()}
    df = df.assign(clave=df["archivo"].map(clave_archivo), actor=df["actor"].map(normalizar_actor),
                   emocion=df["emocion"].map(normalizar_etiqueta))
    n0 = len(df)
    df = df[df["emocion"].isin(EMOCIONES)].drop_duplicates("clave")
    df = df[df["clave"].isin(fisicas)]
    df = df.assign(ruta=df["clave"].map(lambda c: str(fisicas[c])))
    videos, origenes = asignar_video(df["clave"], leer_registros_grupos(carpeta_curso))
    df["video"] = df["actor"] + "|" + pd.Series(videos, index=df.index).astype(str)   # video calificado con el actor
    df["origen_video"] = origenes
    print(f"filas en el CSV {n0} · imágenes utilizables {len(df)}")
    return df.sort_values("clave").reset_index(drop=True)[["archivo", "clave", "ruta", "actor", "emocion", "video", "origen_video"]]


df = construir_registro(DATASET, CARPETA_CURSO)
HUELLA = huella_dataset(df)
print(f"huella del dataset {HUELLA} · actores {df['actor'].nunique()} · videos {df['video'].nunique()}")
print("origen del video:", df["origen_video"].value_counts().to_dict())
# Sin los registros de los grupos se pierden videos (129 en vez de 154) y la partición cambia: se frena acá.
assert (df["origen_video"] != "nombre de archivo").any(), \
    "No se leyeron los registros de video de los grupos: revisar el acceso directo a la carpeta del curso."

In [ ]:
# 2.2 Distribución actor × emoción
display(pd.crosstab(df["actor"], df["emocion"], margins=True, margins_name="Total"))
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={"width_ratios": [1, 2]})
df["emocion"].value_counts().reindex(EMOCIONES).plot.bar(ax=axes[0], color="steelblue", title="Imágenes por emoción")
df["actor"].value_counts().plot.bar(ax=axes[1], color="slategray", title="Imágenes por actor")
plt.tight_layout(); plt.show()

# 2.3 Ejemplos: una fila por emoción
fig, axes = plt.subplots(len(EMOCIONES), 8, figsize=(14, 9))
for f, emo in enumerate(EMOCIONES):
    for c, (_, fila) in enumerate(df[df["emocion"] == emo].sample(8, random_state=SEED).iterrows()):
        axes[f, c].imshow(Image.open(fila["ruta"])); axes[f, c].axis("off")
        axes[f, c].set_title(("" if c else f"{emo.upper()}\n") + fila["actor"].split("_")[0], fontsize=7, loc="left" if not c else "center")
plt.suptitle("Ejemplos por emoción"); plt.tight_layout(); plt.show()

**Lectura:** emociones con desbalance moderado (*neutral* la más frecuente, *tristeza* la más escasa) y actores muy desparejos (Francella 419 imágenes, Julio Chávez 71). Por eso se reporta **F1 macro** además de accuracy y se entrena con **pesos por clase** (NB08). Techo realista para emoción: el acuerdo entre etiquetadores de nuestro control cruzado fue **42,6 %**.

## 3. Partición por video
Dos cuadros del mismo plano son casi idénticos: si se reparten entre train y test, el modelo reconoce el plano y no el gesto. Reglas: **ningún video de train aparece en validación ni en test** (se verifica con `assert`); se parte por actor buscando ~70/15/15; si un actor no tiene videos suficientes, val y test comparten **un** video cortado en dos mitades (concesión R4, documentada); train nunca comparte. Es determinista (empates con hash + semilla).

### 3.1 Unificar partes de un mismo video
El control de fugas (3.3) mostró que los nombres de video no alcanzan: la planilla de G7 anota como videos distintos las **partes** de un mismo clip (`video_1`, `video_1_p2`, `video_1_p3` de Momo) y la de G6 numera como videos distintos **cortes seguidos de la misma escena** (`Oscar_Martinez11` y `Oscar_Martinez13`). Con la partición vieja, cuadros casi idénticos quedaban en train y en test. Antes de partir se unifican:
1. **Partes `_p2`, `_p3`…** de un mismo video → un solo video.
2. **Por contenido:** dentro de cada actor, dos videos con alguna pareja de fotos con similitud > `UMBRAL_MISMO_VIDEO` (0,82: correlación de píxeles en gris 64×64) pasan a ser uno solo, y así en cadena. Es conservador: puede juntar dos videos distintos pero parecidos (menos flexibilidad para partir), nunca separa uno.

In [ ]:
# 3.1 Unificar partes y videos casi idénticos (antes de partir)
def cargar_pixeles(rutas, tam=IMG_SIZE_BASELINE):
    # Imagen en gris reducida a tam×tam, como vector de 0 a 1 (la usan 3.1, 3.3 y los baselines de la sección 4).
    X = np.empty((len(rutas), tam * tam), dtype=np.float32)
    for i, r in enumerate(rutas):
        with Image.open(r) as im:
            X[i] = np.asarray(im.convert("L").resize((tam, tam), Image.BILINEAR), dtype=np.float32).ravel() / 255.0
    return X


X_pix = cargar_pixeles(df["ruta"])
V = X_pix - X_pix.mean(1, keepdims=True)
V /= np.linalg.norm(V, axis=1, keepdims=True) + 1e-8        # V @ V.T = correlación entre fotos (1 = idénticas)

df["video_original"] = df["video"]
df["video"] = df["video"].str.replace(r"_p\d+$", "", regex=True)      # 1) partes del mismo video
antes = df.groupby("actor")["video"].nunique()
uniones = []
for actor, idx in df.groupby("actor").indices.items():                # 2) por contenido, dentro de cada actor
    vids = df["video"].values[idx]
    padre = {v: v for v in set(vids)}
    def raiz(v):
        while padre[v] != v:
            padre[v] = padre[padre[v]]; v = padre[v]
        return v
    S_act = V[idx] @ V[idx].T
    for a, b in zip(*np.where(np.triu(S_act > UMBRAL_MISMO_VIDEO, 1))):
        ra, rb = raiz(vids[a]), raiz(vids[b])
        if ra != rb:
            padre[rb] = ra
            uniones.append({"actor": actor, "video": vids[a].split("|")[1], "se une con": vids[b].split("|")[1],
                            "similitud": float(S_act[a, b])})
    df.loc[df.index[idx], "video"] = [raiz(v) for v in vids]

print(f"Videos: {df['video_original'].nunique()} según planillas y nombres → {df['video'].nunique()} después de unificar")
display(pd.DataFrame({"videos antes": df.groupby("actor")["video_original"].nunique(), "después de _pN": antes,
                      "después de unificar": df.groupby("actor")["video"].nunique()}).query("`videos antes` != `después de unificar`"))
if uniones:
    display(pd.DataFrame(uniones).style.format({"similitud": "{:.3f}"}).set_caption("Videos unificados por contenido"))

In [ ]:
# 3.2 Partición del dataset (por video)
SPLITS = ("train", "val", "test")
MAX_EXHAUSTIVO, MIN_FRAC_EVAL, PENALIDAD_R4 = 16, 0.35, 0.10


def _h(nombre):
    return hashlib.sha256(f"{SEED}:{nombre}".encode()).hexdigest()


def _separable_por_video(tamanios):
    n, total = len(tamanios), tamanios.sum()
    if n < 2:
        return False
    if n > MAX_EXHAUSTIVO:
        return True
    v = tamanios.to_numpy()
    return any(min(s, total - s) >= MIN_FRAC_EVAL * total
               for s in (sum(v[i] for i in range(n) if m >> i & 1) for m in range(1, 2 ** n - 1)))


def _elegir_train(tamanios):
    # Por actor: conjunto de videos cuyo total se acerque más al 70 %, prefiriendo dejar para evaluación
    # videos separables entre val y test (búsqueda exhaustiva con ≤ 16 videos).
    videos = sorted(tamanios.index, key=lambda v: (-tamanios[v], _h(v)))
    total = tamanios.sum(); objetivo = FRACCIONES["train"] * total
    if len(videos) <= MAX_EXHAUSTIVO:
        mejor, mejor_costo = None, np.inf
        for m in range(1, 2 ** len(videos) - 1):
            elegidos = [v for i, v in enumerate(videos) if m >> i & 1]
            costo = abs(tamanios[elegidos].sum() - objetivo) / total
            if not _separable_por_video(tamanios.drop(elegidos)):
                costo += PENALIDAD_R4
            if costo < mejor_costo:
                mejor, mejor_costo = set(elegidos), costo
        return mejor
    elegidos, acum = set(), 0
    for v in videos:
        if acum + tamanios[v] <= objetivo or not elegidos:
            elegidos.add(v); acum += tamanios[v]
    return elegidos - ({videos[-1]} if len(elegidos) == len(videos) else set())


def _repartir_evaluacion(g):
    tamanios = g.groupby("video").size()
    videos = sorted(tamanios.index, key=_h); total = tamanios.sum()
    mejor, mejor_costo = None, np.inf
    if 2 <= len(videos) <= MAX_EXHAUSTIVO:
        for m in range(1, 2 ** len(videos) - 1):
            a_val = [v for i, v in enumerate(videos) if m >> i & 1]
            n_val = tamanios[a_val].sum()
            if min(n_val, total - n_val) >= MIN_FRAC_EVAL * total and abs(n_val - total / 2) < mejor_costo:
                mejor, mejor_costo = set(a_val), abs(n_val - total / 2)
    if mejor is not None:                                     # por video
        return pd.Series(np.where(g["video"].isin(mejor), "val", "test"), index=g.index)
    orden = g.assign(_hv=g["video"].map(_h)).sort_values(["_hv", "clave"]).index   # R4: mitades contiguas
    mitad = len(orden) // 2
    return pd.Series(["val"] * mitad + ["test"] * (len(orden) - mitad), index=orden).reindex(g.index)


def particionar(df):
    split, avisos = pd.Series(index=df.index, dtype=object), []
    for actor, g in df.groupby("actor", sort=True):
        tamanios = g.groupby("video").size()
        if len(tamanios) == 1:                                # un solo video: todo a train
            split.loc[g.index] = "train"; avisos.append(f"{actor}: un solo video -> todo a train"); continue
        es_train = g["video"].isin(_elegir_train(tamanios))
        split.loc[g.index[es_train]] = "train"
        ev = _repartir_evaluacion(g[~es_train]); split.loc[ev.index] = ev
        compartidos = set(g.loc[ev.index[ev == "val"], "video"]) & set(g.loc[ev.index[ev == "test"], "video"])
        if compartidos:
            avisos.append(f"{actor}: val y test comparten {sorted(compartidos)} (R4)")
    return split, avisos


def verificar_sin_fugas(df, split):
    vids = {s: set(df.loc[split == s, "video"]) for s in SPLITS}
    fuga = vids["train"] & (vids["val"] | vids["test"])
    assert not fuga, f"FUGA: {sorted(fuga)[:5]}"
    assert split.notna().all(), "Hay imágenes sin split"
    assert set(df["actor"]) == set(df.loc[split == "train", "actor"]), "Hay actores sin train"


split, avisos = particionar(df)
verificar_sin_fugas(df, split)
assert split.equals(particionar(df)[0]), "La partición no es determinista"
df["split"] = split
print("✔ Sin fugas: ningún video de train aparece en validación o test. ✔ Determinista.")
display(pd.DataFrame({s: {"imágenes": int((split == s).sum()), "videos": df.loc[split == s, "video"].nunique(),
                          "actores": df.loc[split == s, "actor"].nunique()} for s in SPLITS}).T)
print("Concesiones R4:"); [print("  -", a) for a in avisos]

### 3.3 Control de fugas
Dos controles independientes:
1. **Por nombre de video:** cuántos videos aparecen en más de un conjunto. Train con val o test tiene que dar **0** (si no, el notebook se frena); val con test son los videos de la concesión R4.
2. **Por contenido de la imagen:** para cada foto de val/test, la foto de train más parecida (correlación de píxeles en gris 64×64). Con la unificación de 3.1 no puede quedar ninguna del mismo actor por encima de `UMBRAL_MISMO_VIDEO` (si queda, el notebook se frena). La tabla resume, por video, las parejas del mismo actor más parecidas que quedan, y los gráficos muestran los 6 pares más parecidos para mirarlos: tienen que ser escenas distintas.

**Historia (D-021, D-022):** en la primera versión de este control, los pares más parecidos eran cuadros del mismo video de Momo (`video_1_p3` en val/test y `video_1`, `video_1_p2` en train, 15 fotos, similitud 0,89) y de Oscar Martínez (cortes 11/13 y 6/8 de la misma escena, 5 fotos, 0,83–0,88). De ahí salió la unificación de 3.1.

In [ ]:
# 3.3 Control de fugas: por nombre de video y por contenido de la imagen
# 1) Por nombre de video
vids = {s: set(df.loc[df["split"] == s, "video"]) for s in SPLITS}
cruces = {"train ∩ validación": vids["train"] & vids["val"], "train ∩ test": vids["train"] & vids["test"],
          "validación ∩ test (R4, declarado)": vids["val"] & vids["test"]}
display(pd.DataFrame({"videos compartidos": {k: len(v) for k, v in cruces.items()}}))
assert not cruces["train ∩ validación"] and not cruces["train ∩ test"], "FUGA: hay videos de train en validación o test"

# 2) Por contenido: la foto de train más parecida a cada foto de validación/test (V viene de 3.1)
i_tr, i_ev = np.where(df["split"] == "train")[0], np.where(df["split"] != "train")[0]
S = V[i_ev] @ V[i_tr].T
gemela = i_tr[S.argmax(1)]
pares = pd.DataFrame({"conjunto": df["split"].values[i_ev], "foto": df["clave"].values[i_ev],
                      "más parecida en train": df["clave"].values[gemela], "similitud": S.max(1),
                      "mismo actor": df["actor"].values[i_ev] == df["actor"].values[gemela]}).sort_values("similitud", ascending=False)
for umbral in (0.95, 0.90, UMBRAL_MISMO_VIDEO, 0.80):
    n = int(((pares["similitud"] > umbral) & pares["mismo actor"]).sum())
    print(f"Fotos de val/test con una foto de train del mismo actor con similitud > {umbral}: {n} de {len(pares)}")
assert not ((pares["similitud"] > UMBRAL_MISMO_VIDEO) & pares["mismo actor"]).any(), "FUGA: quedan fotos casi idénticas en train y val/test"

video_de = df.set_index("clave")["video"]
resumen = (pares[pares["mismo actor"]].assign(**{"video val/test": lambda d: d["foto"].map(video_de),
                                                 "video train": lambda d: d["más parecida en train"].map(video_de)})
           .groupby(["video val/test", "video train"]).agg(fotos=("foto", "count"), similitud_max=("similitud", "max"))
           .sort_values("similitud_max", ascending=False).head(10))
display(resumen.style.format({"similitud_max": "{:.3f}"}).set_caption("Parejas de videos (mismo actor) más parecidas entre val/test y train"))
display(pares.head(10).style.format({"similitud": "{:.3f}"}))

top = pares.head(6)
fig, axes = plt.subplots(2, len(top), figsize=(2.3 * len(top), 5))
for k, (_, f) in enumerate(top.iterrows()):
    for fila, clave in enumerate((f["foto"], f["más parecida en train"])):
        axes[fila, k].imshow(Image.open(df.loc[df["clave"] == clave, "ruta"].iloc[0])); axes[fila, k].axis("off")
    axes[0, k].set_title(f"{f['conjunto']} · {f['similitud']:.3f}", fontsize=9); axes[1, k].set_title("train", fontsize=9)
fig.suptitle("Los 6 pares más parecidos entre val/test (arriba) y train (abajo)"); plt.tight_layout(); plt.show()

## 4. Baselines (el piso)
Ninguna red se justifica sin un baseline (NB01): el **trivial** (clase más frecuente), un **Random Forest** y un **MLP de scikit-learn** sobre los píxeles en gris reducidos a 64×64 (4.096 columnas; con 224×224×3 = 150.528 columnas y ~1.900 imágenes no hay de dónde generalizar). Se miden en **validación**; el test queda reservado.

In [ ]:
# 4.1 Baselines en validación
def metricas(y_true, y_pred):
    # F1 macro pesa igual a cada clase: con clases desbalanceadas es la métrica honesta.
    return {"accuracy": accuracy_score(y_true, y_pred), "f1_macro": f1_score(y_true, y_pred, average="macro")}


def a_enteros(*ys):
    # scikit-learn 1.6 (Colab) falla en MLPClassifier(early_stopping=True) con etiquetas de texto.
    clases = {c: i for i, c in enumerate(np.unique(np.concatenate([np.asarray(y) for y in ys])))}
    return [np.array([clases[v] for v in y]) for y in ys]


def mlp(arq=(256,), **kw):
    return MLPClassifier(hidden_layer_sizes=arq, random_state=SEED, **({"early_stopping": True, "max_iter": 200} | kw))


m_tr, m_va = (df["split"] == "train").to_numpy(), (df["split"] == "val").to_numpy()
resultados_baseline = {}
for tarea in TAREAS:
    filas = []
    y_tr, y_va = a_enteros(df.loc[m_tr, tarea], df.loc[m_va, tarea])
    for nombre, m in {"Trivial (clase más frecuente)": DummyClassifier(strategy="most_frequent"),
                      "Random Forest (píxeles 64×64)": RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                                                              n_jobs=-1, random_state=SEED),
                      "MLP scikit-learn (píxeles 64×64)": mlp()}.items():
        m.fit(X_pix[m_tr], y_tr)
        filas.append({"modelo": nombre, **metricas(y_va, m.predict(X_pix[m_va]))})
    resultados_baseline[tarea] = pd.DataFrame(filas).set_index("modelo")
    print(f"=== {tarea.upper()} (validación)"); display(resultados_baseline[tarea].round(3))

## 5. Acta de límites (NB02)
1. **Costo:** al agrandar el MLP, el tiempo crece mucho más rápido que el beneficio.
2. **Permutación de píxeles:** se mezclan los píxeles de *todas* las imágenes con una misma permutación fija; para una persona quedan irreconocibles. Si el MLP rinde igual, nunca usó la estructura espacial de la cara: la trata como 4.096 números sueltos.

In [ ]:
# 5.1 Mediciones del acta (identidad para el costo; las dos tareas para la permutación)
PERM_BASELINE = np.random.default_rng(SEED).permutation(IMG_SIZE_BASELINE ** 2)
y_tr, y_va = a_enteros(df.loc[m_tr, "actor"], df.loc[m_va, "actor"])
filas = []
for arq in ((64,), (256,), (512, 256), (1024, 512)):
    m = mlp(arq, early_stopping=False, max_iter=15)
    t0 = time.perf_counter(); m.fit(X_pix[m_tr], y_tr)
    filas.append({"arquitectura": str(arq), "parámetros": sum(w.size for w in m.coefs_) + sum(b.size for b in m.intercepts_),
                  "segundos": round(time.perf_counter() - t0, 1), "accuracy val": accuracy_score(y_va, m.predict(X_pix[m_va]))})
display(pd.DataFrame(filas).set_index("arquitectura"))

perm_mlp = {}
for tarea in TAREAS:
    y_tr, y_va = a_enteros(df.loc[m_tr, tarea], df.loc[m_va, tarea])
    perm_mlp[tarea] = {nombre: accuracy_score(y_va, mlp().fit(X[m_tr], y_tr).predict(X[m_va]))
                       for nombre, X in (("original", X_pix), ("píxeles permutados", X_pix[:, PERM_BASELINE]))}
display(pd.DataFrame(perm_mlp).T.add_prefix("accuracy val · "))

**Acta:** si el MLP rinde prácticamente igual con los píxeles destruidos, no ve una imagen. Requisitos para la herramienta siguiente: **capas convolucionales** (comparten pesos sobre vecindades y dependen de la estructura espacial; se comprueba en 7.2), **GPU** y **callbacks** para controlar el entrenamiento.

## 6. Pipeline `tf.data`, aumentación y herramientas de entrenamiento
Patrón del curso: `map(leer) → cache → shuffle (solo train) → batch → prefetch`. Las imágenes llegan al modelo en [0, 255]; **escalado y aumentación van como capas dentro del modelo** (clase 17), así una imagen nueva pasa exactamente por el mismo preprocesamiento.

| Aumentación | ¿Válida para rostros? |
|---|---|
| Flip horizontal | Sí: una cara espejada es la misma persona con el mismo gesto |
| Rotación ±14° y zoom ±10 % | Sí: cabezas inclinadas, recortes no centrados |
| Brillo / contraste ±10 % | Sí: fuentes con iluminación distinta |
| Flip vertical, rotaciones grandes | **No**: fabrican caras imposibles (KerasTuner confirmó que la aumentación fuerte empeora) |

**Early stopping sobre el accuracy de validación**, no sobre `val_loss` (D-017): con pocos videos la red se vuelve sobreconfiada y la pérdida de validación sube aunque acierte más; en la primera búsqueda eso llevó a elegir un modelo con 6,5 % de accuracy.

In [ ]:
# 6.1 Datos, aumentación, entrenamiento y evaluación (se usan para la CNN y para ResNet50)
AUTOTUNE = tf.data.AUTOTUNE


def _leer_imagen(ruta, tam):
    img = tf.io.decode_image(tf.io.read_file(ruta), channels=3, expand_animations=False)
    img = tf.cast(tf.round(tf.image.resize(img, (tam, tam))), tf.uint8)
    img.set_shape((tam, tam, 3))
    return img


def datasets_por_split(tarea, tam, batch, permutacion=None):
    # Vocabulario de clases fijo (tomado de train); etiquetas enteras; un tf.data por split.
    # `permutacion`: orden fijo de las posiciones de píxel (experimento 7.2); los 3 canales viajan juntos.
    clases = sorted(df.loc[df["split"] == "train", tarea].unique())
    indice = {c: i for i, c in enumerate(clases)}
    ys, ds = {}, {}
    for s in SPLITS:
        d = df[df["split"] == s]
        ys[s] = d[tarea].map(indice).to_numpy(dtype=np.int32)
        x = tf.data.Dataset.from_tensor_slices((d["ruta"].to_numpy(dtype=str), ys[s]))
        x = x.map(lambda r, y: (_leer_imagen(r, tam), y), num_parallel_calls=AUTOTUNE)
        if permutacion is not None:
            p = tf.constant(permutacion, dtype=tf.int32)
            x = x.map(lambda i, y: (tf.reshape(tf.gather(tf.reshape(i, (-1, 3)), p), (tam, tam, 3)), y), num_parallel_calls=AUTOTUNE)
        x = x.cache()
        if s == "train":
            x = x.shuffle(len(d), seed=SEED, reshuffle_each_iteration=True)
        ds[s] = x.map(lambda i, y: (tf.cast(i, tf.float32), y), num_parallel_calls=AUTOTUNE).batch(batch).prefetch(AUTOTUNE)
    return ds, clases, ys


def pesos_por_clase(y, k):
    # Pesos 'balanced' n / (K · n_c) (NB08): un error en una clase chica pesa más.
    conteo = np.bincount(y, minlength=k)
    return {i: float(len(y) / (k * c)) for i, c in enumerate(conteo) if c}


def capas_aumentacion(rotacion, zoom, color, traslacion=0.06):
    # Invarianzas válidas para rostros (NB03, NB16, clase 17). Solo actúan en entrenamiento.
    return keras.Sequential([
        layers.RandomFlip("horizontal", seed=SEED),
        layers.RandomRotation(rotacion, fill_mode="nearest", seed=SEED),
        layers.RandomZoom(zoom, fill_mode="nearest", seed=SEED),
        layers.RandomTranslation(traslacion, traslacion, fill_mode="nearest", seed=SEED),
        layers.RandomBrightness(color, value_range=(0, 255), seed=SEED),
        layers.RandomContrast(color, seed=SEED),
    ], name="aumentacion")


def entrenar(modelo, ds, lr, epocas, paciencia, pesos, desde=0):
    # Adam + early stopping por ACCURACY de validación con los mejores pesos (NB07, NB09, D-017).
    modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    cb = [keras.callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=paciencia,
                                        restore_best_weights=True, start_from_epoch=desde)]
    return modelo.fit(ds["train"], validation_data=ds["val"], epochs=epocas, class_weight=pesos, callbacks=cb, verbose=0)


def evaluar(modelo, ds, ys, split):
    prob = modelo.predict(ds[split], verbose=0)
    return metricas(ys[split], prob.argmax(1)), prob


def graficar_curvas(historias, titulo):
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.6)); x0 = 0
    for nombre, h in historias.items():
        ep = np.arange(len(h.history["loss"])) + x0
        for ax, met in zip(axes, ("loss", "accuracy")):
            ax.plot(ep, h.history[met], label=f"train {nombre}"); ax.plot(ep, h.history[f"val_{met}"], "--", label=f"val {nombre}")
        x0 += len(h.history["loss"])
    for ax, met in zip(axes, ("pérdida", "accuracy")):
        ax.set_title(f"{titulo} — {met}"); ax.set_xlabel("época"); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()


modelos = {}     # (modelo, tarea) -> dict con el modelo, sus datos y resultados

# Una imagen de train y 7 versiones aumentadas (con la aumentación de la CNN final)
aug = capas_aumentacion(HP_CNN["rotacion"], HP_CNN["zoom"], HP_CNN["color"])
img = np.asarray(Image.open(df.loc[df["split"] == "train", "ruta"].iloc[0]).convert("RGB"), dtype="float32")
fig, axes = plt.subplots(1, 8, figsize=(16, 2.4))
axes[0].imshow(img.astype("uint8")); axes[0].set_title("original")
for k in range(1, 8):
    axes[k].imshow(np.clip(aug(img[None], training=True)[0].numpy(), 0, 255).astype("uint8")); axes[k].set_title(f"aumentada {k}")
for a in axes:
    a.axis("off")
plt.tight_layout(); plt.show()

## 7. CNN propia final (entrenada desde cero)
Arquitectura residual de NB13, **una red por tarea** (una para identidad y otra para emoción), entrada 112×112. Cómo se llegó a esta versión (las versiones anteriores, documentadas en el **Anexo A**):

| Paso | Cambio | Identidad val (acc / F1) | Emoción val (acc / F1) |
|---|---|---|---|
| v1 (control, NB11) | 4 bloques Conv-Conv-BN-Pool, 224×224 | 0,287 / 0,249 | 0,264 / 0,256 |
| v3 · E14 | residual, dos salidas, KerasTuner eligiendo por `val_loss` | 0,252 / 0,224 | 0,290 / 0,270 |
| v3 · E15 | KerasTuner eligiendo por **accuracy** (D-017) | 0,355 / 0,281 | 0,350 / 0,305 |
| **v3 · E16** | búsqueda afinada + **una red por tarea** (D-018) | **0,414 / 0,361** | **0,393 / 0,349** |

| Componente | Dónde se vio | Por qué |
|---|---|---|
| Bloques residuales `x + F(x)` | NB13, clase 7 | Más profundidad sin que el gradiente se pierda |
| Una red por tarea | NB11 (v1), clase 10 | Con dos salidas la emoción competía con la identidad por el mismo tronco (+6 puntos en identidad al separarlas) |
| Entrada 112×112 | decisión de datos | Menos detalle para memorizar el video |
| Aumentación como capas | NB03, NB16, clase 17 | Invarianzas válidas para rostros |
| BatchNorm, Dropout 0,4 | NB07 | Regularización |
| Pesos por clase | NB08 | Desbalance de actores y emociones |
| Adam lr 1,4e-3 + early stopping por accuracy de validación | NB06, NB07, NB09 | Hiperparámetros elegidos por KerasTuner |

In [ ]:
# 7.1 Arquitectura y entrenamiento (una red por tarea)
def bloque_residual(x, filtros, paso, nombre, reg=None):
    # Bloque residual (NB13): la salida es x + F(x); si cambia la forma, el atajo es una conv 1×1.
    atajo = x
    x = layers.Conv2D(filtros, 3, strides=paso, padding="same", use_bias=False, kernel_initializer="he_normal",
                      kernel_regularizer=reg, name=f"{nombre}_conv1")(x)
    x = layers.ReLU()(layers.BatchNormalization(momentum=0.9, name=f"{nombre}_bn1")(x))
    x = layers.Conv2D(filtros, 3, padding="same", use_bias=False, kernel_initializer="he_normal",
                      kernel_regularizer=reg, name=f"{nombre}_conv2")(x)
    x = layers.BatchNormalization(momentum=0.9, name=f"{nombre}_bn2")(x)
    if paso != 1 or atajo.shape[-1] != filtros:
        atajo = layers.Conv2D(filtros, 1, strides=paso, use_bias=False, kernel_regularizer=reg, name=f"{nombre}_atajo")(atajo)
        atajo = layers.BatchNormalization(momentum=0.9, name=f"{nombre}_atajo_bn")(atajo)
    return layers.ReLU()(layers.Add()([x, atajo]))


def cnn_propia(n_clases, nombre, hp=HP_CNN, tam=IMG_SIZE_CNN, aumentar=True):
    # Tallo + bloque a resolución completa + 3 bloques que reducen a la mitad y duplican filtros
    # (menos "dónde", más "qué") + GlobalAveragePooling + Dropout + softmax.
    fijar_semillas()
    reg = keras.regularizers.l2(hp["l2"]) if hp["l2"] else None
    base = hp["base"]
    entrada = keras.Input(shape=(tam, tam, 3))
    x = capas_aumentacion(hp["rotacion"], hp["zoom"], hp["color"])(entrada) if aumentar else entrada
    x = layers.Rescaling(1 / 127.5, offset=-1)(x)
    x = layers.Conv2D(base, 3, padding="same", use_bias=False, kernel_initializer="he_normal", kernel_regularizer=reg)(x)
    x = layers.ReLU()(layers.BatchNormalization(momentum=0.9)(x))
    x = bloque_residual(x, base, 1, "r0", reg)
    x = layers.MaxPooling2D()(x)                                 # 112 → 56
    for i, f in enumerate((2 * base, 4 * base, 8 * base), 1):
        x = bloque_residual(x, f, 2, f"r{i}", reg)               # 28 → 14 → 7
    x = layers.Dropout(hp["dropout"])(layers.GlobalAveragePooling2D()(x))
    return keras.Model(entrada, layers.Dense(n_clases, activation="softmax")(x), name=nombre)


for tarea in TAREAS:
    t0 = time.time()
    ds, clases, ys = datasets_por_split(tarea, IMG_SIZE_CNN, BATCH_CNN)
    m = cnn_propia(len(clases), f"cnn_propia_{tarea}")
    h = entrenar(m, ds, HP_CNN["lr"], EPOCAS_MAX_CNN, PACIENCIA_CNN, pesos_por_clase(ys["train"], len(clases)), desde=10)
    val, _ = evaluar(m, ds, ys, "val")
    modelos[("cnn", tarea)] = {"nombre": "CNN propia", "modelo": m, "ds": ds, "clases": clases, "ys": ys, "val": val}
    graficar_curvas({"": h}, f"CNN propia — {tarea}")
    print(f"{tarea}: VAL accuracy={val['accuracy']:.3f} · F1 macro={val['f1_macro']:.3f} · "
          f"{m.count_params():,} parámetros · {len(h.history['loss'])} épocas · {time.time() - t0:.0f} s")

### 7.2 La revancha de la permutación (NB11)
Misma CNN, mismos hiperparámetros, pero con los píxeles permutados (sin aumentación: rotar o espejar una imagen permutada no tiene sentido). Si la CNN **sí** usa la estructura espacial, su desempeño se tiene que desplomar: lo opuesto a lo que le pasó al MLP en el acta.

In [ ]:
# 7.2 CNN con píxeles permutados (identidad)
ds_p, clases_p, ys_p = datasets_por_split("actor", IMG_SIZE_CNN, BATCH_CNN,
                                          permutacion=np.random.default_rng(SEED).permutation(IMG_SIZE_CNN ** 2))
cnn_perm = cnn_propia(len(clases_p), "cnn_permutada", aumentar=False)
entrenar(cnn_perm, ds_p, HP_CNN["lr"], 60, 10, pesos_por_clase(ys_p["train"], len(clases_p)), desde=10)
display(pd.DataFrame({"accuracy val · original": [perm_mlp["actor"]["original"], modelos[("cnn", "actor")]["val"]["accuracy"]],
                      "accuracy val · píxeles permutados": [perm_mlp["actor"]["píxeles permutados"], evaluar(cnn_perm, ds_p, ys_p, "val")[0]["accuracy"]]},
                     index=["MLP scikit-learn", "CNN propia"]))

## 8. ResNet50 preentrenada (transfer learning)
Los recortes ya son 224×224×3, la entrada nativa de ResNet50, y con ~90 imágenes por actor no alcanza para aprender filtros desde cero. Secuencia del NB15:
- **Fase 1:** base congelada; solo se entrena la cabeza nueva (`GAP → Dropout(0,3) → softmax`) con Adam 1e-3.
- **Fase 2:** se **descongelan** los bloques conv4 y conv5 (vuelven a entrenarse para adaptarse a rostros; las BatchNorm siguen congeladas) y se recompila con Adam **1e-5**, 100 veces menor, para no destruir lo aprendido. En el notebook completo se comparó contra descongelar solo conv5: conv4 + conv5 fue mejor en las dos tareas.

In [ ]:
# 8.1 ResNet50 en dos fases
def resnet50_transferencia(n_clases, nombre):
    fijar_semillas()
    base = keras.applications.ResNet50(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    entrada = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = PreprocesoResNet50()(capas_aumentacion(0.022, 0.10, 0.15, traslacion=0.05)(entrada))
    x = base(x, training=False)          # BatchNorm en modo inferencia aunque luego se descongele
    x = layers.Dropout(0.3)(layers.GlobalAveragePooling2D()(x))
    return keras.Model(entrada, layers.Dense(n_clases, activation="softmax")(x), name=nombre), base


def descongelar_desde(base, primera):
    # Fase 2: desde `primera` en adelante las capas vuelven a entrenarse; las BatchNorm siguen congeladas.
    base.trainable = True
    corte = [l.name for l in base.layers].index(primera)
    for i, capa in enumerate(base.layers):
        capa.trainable = i >= corte and not isinstance(capa, layers.BatchNormalization)


for tarea in TAREAS:
    t0 = time.time()
    ds, clases, ys = datasets_por_split(tarea, IMG_SIZE, BATCH_RESNET)
    pesos = pesos_por_clase(ys["train"], len(clases))
    m, base = resnet50_transferencia(len(clases), f"resnet50_{tarea}")
    h1 = entrenar(m, ds, 1e-3, 20, 5, pesos)
    val1, _ = evaluar(m, ds, ys, "val")
    descongelar_desde(base, RESNET_PRIMERA_DESCONGELADA)        # + recompilar dentro de entrenar()
    h2 = entrenar(m, ds, 1e-5, 20, 5, pesos)
    val, _ = evaluar(m, ds, ys, "val")
    modelos[("resnet", tarea)] = {"nombre": "ResNet50 (conv4+conv5)", "modelo": m, "ds": ds, "clases": clases, "ys": ys,
                                  "val": val, "val fase 1": val1}
    graficar_curvas({"fase 1": h1, "fase 2": h2}, f"ResNet50 — {tarea}")
    print(f"{tarea}: VAL fase 1 acc={val1['accuracy']:.3f} F1={val1['f1_macro']:.3f} → "
          f"fase 2 acc={val['accuracy']:.3f} F1={val['f1_macro']:.3f} · {time.time() - t0:.0f} s")

## 9. Comparación en validación
Acá se toman las decisiones. La tabla por actor muestra **para quién funciona** cada modelo y conecta con el desbalance: el profesor habilitó excluir actores que ningún modelo reconoce, pero el único caso (Eva De Dominici) tiene 8 videos, no le faltan datos, así que **no se excluyó a nadie** (D-018).

In [ ]:
# 9.1 Tabla de validación y recall por actor
filas = []
for tarea in TAREAS:
    for nombre, fila in resultados_baseline[tarea].iterrows():
        filas.append({"tarea": tarea, "modelo": nombre, **fila.to_dict()})
    filas.append({"tarea": tarea, "modelo": "CNN propia (desde cero)", **modelos[("cnn", tarea)]["val"]})
    filas.append({"tarea": tarea, "modelo": "ResNet50 fase 1 (base congelada)", **modelos[("resnet", tarea)]["val fase 1"]})
    filas.append({"tarea": tarea, "modelo": "ResNet50 fase 2 (conv4+conv5)", **modelos[("resnet", tarea)]["val"]})
display(pd.DataFrame(filas).set_index(["tarea", "modelo"]).style.format("{:.3f}"))


def recall_por_clase(r, split="val"):
    pred = r["modelo"].predict(r["ds"][split], verbose=0).argmax(1)
    cm = confusion_matrix(r["ys"][split], pred, labels=range(len(r["clases"])))
    return pd.Series(np.diag(cm) / np.maximum(cm.sum(1), 1), index=r["clases"])


val = df[df["split"] == "val"]
display(pd.DataFrame({"CNN propia · recall": recall_por_clase(modelos[("cnn", "actor")]),
                      "ResNet50 · recall": recall_por_clase(modelos[("resnet", "actor")]),
                      "imágenes train": df.loc[df["split"] == "train", "actor"].value_counts(),
                      "imágenes val": val["actor"].value_counts(),
                      "videos": df.groupby("actor")["video"].nunique()})
        .sort_values("CNN propia · recall").style.format({"CNN propia · recall": "{:.2f}", "ResNet50 · recall": "{:.2f}"}))

## 10. Evaluación final en test — una sola vez
Todas las decisiones se tomaron con validación. Ahora se evalúan en test los modelos finales; esta celda no se vuelve a correr para "probar otra cosa".

In [ ]:
# 10.1 Resultados en test y matrices de confusión (en la diagonal, los aciertos de cada clase)
resultados_test = {}
for clave, r in modelos.items():
    met, prob = evaluar(r["modelo"], r["ds"], r["ys"], "test")
    resultados_test[clave] = {"metricas": met, "prob": prob}
display(pd.DataFrame([{"tarea": t, "modelo": modelos[(m, t)]["nombre"], **resultados_test[(m, t)]["metricas"]}
                      for (m, t) in modelos]).set_index(["tarea", "modelo"]).style.format("{:.3f}"))
print(f"Huella del dataset: {HUELLA} · semilla: {SEED}")

for clave in [("cnn", "actor"), ("cnn", "emocion"), ("resnet", "actor"), ("resnet", "emocion")]:
    r, pred = modelos[clave], resultados_test[clave]["prob"].argmax(1)
    print(f"\n===== {r['nombre']} · {clave[1]} (test) =====")
    print(classification_report(r["ys"]["test"], pred, labels=range(len(r["clases"])), target_names=r["clases"], zero_division=0))
    lado = max(5, 0.45 * len(r["clases"]) + 2)
    fig, ax = plt.subplots(figsize=(lado, lado))
    cm = confusion_matrix(r["ys"]["test"], pred, labels=range(len(r["clases"])), normalize="true")
    ConfusionMatrixDisplay(cm, display_labels=r["clases"]).plot(ax=ax, cmap="Blues", colorbar=False, values_format=".2f", xticks_rotation=90)
    ax.set_title(f"{r['nombre']} — {clave[1]} (test, normalizada por fila)"); plt.tight_layout(); plt.show()
    cm[np.diag_indices_from(cm)] = 0                            # pares más confundidos (fuera de la diagonal)
    i, j = np.unravel_index(np.argsort(-cm.ravel())[:5], cm.shape)
    print("Pares más confundidos (real → predicho, % de la fila):",
          " · ".join(f"{r['clases'][a]} → {r['clases'][b]} {cm[a, b]:.0%}" for a, b in zip(i, j) if cm[a, b]))

In [ ]:
# 10.2 Errores con mayor confianza: ¿razonables o absurdos?
rutas_test = df.loc[df["split"] == "test", "ruta"].to_numpy()
for clave in [("cnn", "actor"), ("cnn", "emocion"), ("resnet", "actor"), ("resnet", "emocion")]:
    r, prob = modelos[clave], resultados_test[clave]["prob"]
    y, pred = r["ys"]["test"], prob.argmax(1)
    err = np.where(pred != y)[0]
    peores = err[np.argsort(-prob[err, pred[err]])][:12]
    fig, axes = plt.subplots(2, 6, figsize=(14, 5.6))
    for ax in axes.ravel():
        ax.axis("off")
    for ax, i in zip(axes.ravel(), peores):
        ax.imshow(Image.open(rutas_test[i]))
        ax.set_title(f"real: {r['clases'][y[i]]}\npred: {r['clases'][pred[i]]} ({prob[i, pred[i]]:.0%})", fontsize=8, color="darkred")
    fig.suptitle(f"Peores errores — {r['nombre']} — {clave[1]}"); plt.tight_layout(); plt.show()

## 11. Interpretación: ¿el modelo mira la cara? (oclusión, NB12)
Se tapa cada región con un parche gris y se mide cuánto cae la probabilidad de la clase correcta: en rojo, las regiones decisivas. Es la prueba del riesgo declarado en el TP1: como cada actor sale de pocos videos, un modelo podría reconocer **el fondo o la serie** en lugar de la cara.

In [ ]:
# 11.1 Mapas de oclusión sobre aciertos de test
def mapa_oclusion(modelo, imagen, clase, gris=127.5):
    lado = imagen.shape[0]; parche, paso = lado // 7, lado // 14
    pos = list(range(0, lado - parche + 1, paso)); lote = []
    for y in pos:
        for x in pos:
            t = imagen.copy(); t[y:y + parche, x:x + parche] = gris; lote.append(t)
    p0 = float(modelo.predict(imagen[None], verbose=0)[0, clase])
    caidas = (p0 - modelo.predict(np.stack(lote), batch_size=64, verbose=0)[:, clase]).reshape(len(pos), len(pos))
    mapa, cuenta = np.zeros((lado, lado)), np.zeros((lado, lado))
    for a, y in enumerate(pos):
        for b, x in enumerate(pos):
            mapa[y:y + parche, x:x + parche] += caidas[a, b]; cuenta[y:y + parche, x:x + parche] += 1
    return mapa / np.maximum(cuenta, 1), p0


for clave in [("cnn", "actor"), ("cnn", "emocion"), ("resnet", "actor"), ("resnet", "emocion")]:
    r, prob = modelos[clave], resultados_test[clave]["prob"]
    lado = r["modelo"].inputs[0].shape[1]                       # CNN 112 · ResNet50 224
    bien = np.where(prob.argmax(1) == r["ys"]["test"])[0]
    elegidas = np.random.default_rng(SEED).choice(bien, size=min(5, len(bien)), replace=False)
    fig, axes = plt.subplots(2, len(elegidas), figsize=(3 * len(elegidas), 6.2))
    for k, i in enumerate(elegidas):
        img = np.asarray(Image.open(rutas_test[i]).convert("RGB").resize((lado, lado), Image.BILINEAR), dtype="float32")
        mapa, p = mapa_oclusion(r["modelo"], img, int(r["ys"]["test"][i]))
        axes[0, k].imshow(img.astype("uint8")); axes[0, k].set_title(f"{r['clases'][r['ys']['test'][i]]} (p={p:.2f})", fontsize=9)
        axes[1, k].imshow(img.astype("uint8")); axes[1, k].imshow(mapa, cmap="jet", alpha=0.5)
        axes[0, k].axis("off"); axes[1, k].axis("off")
    fig.suptitle(f"Oclusión — {r['nombre']} — {clave[1]} (rojo = región decisiva)"); plt.tight_layout(); plt.show()

## 12. Predicción sobre fotos nuevas
Se sube cualquier foto de uno de los 20 actores y las dos redes dicen quién es y qué emoción expresa, con su porcentaje de confianza (la probabilidad de la salida softmax). Se muestran las dos redes: la CNN propia (desde cero) y ResNet50 (transfer learning).

**Dos formas de usarla:**
- **Con las redes recién entrenadas** («Ejecutar todas»): 12.1 guarda las cuatro redes y descarga `modelos_g6.zip` a tu computadora (unos 600 MB; si el navegador no lo baja solo, descargalo desde el panel de archivos); 12.2 ve que ya están en memoria y no hace nada; 12.4 predice.
- **Sin volver a entrenar** (otra sesión): correr **solo la celda 1.1**, después **12.2** (carga `modelos_g6.zip`: lo pide para subir, o lo busca en `/content` o en la raíz de tu Mi unidad), **12.3** y **12.4**. Tarda un par de minutos en lugar de 55.
- **Las fotos** van en la carpeta `/content/fotos_nuevas` (jpg, png o webp, mejor recortadas a la cara); 12.4 predice todas y muestra un resumen al final.

**Para que la predicción tenga sentido:** las redes se entrenaron con **recortes de rostro**, así que la foto se recorta a un cuadrado central y conviene subirla ya recortada a la cara. Solo conocen a estos 20 actores y 5 emociones: con una persona que no está, igual eligen a alguien de la lista.

In [ ]:
# 12.1 Guardar las cuatro redes entrenadas y descargarlas (después de "Ejecutar todas")
import json, zipfile
from google.colab import files

GUARDAR_Y_DESCARGAR = True                     # False: no guarda ni descarga
CARPETA_MODELOS = Path("/content/modelos_g6")
ZIP_MODELOS = Path("/content/modelos_g6.zip")

if GUARDAR_Y_DESCARGAR and "modelos" in globals() and ("cnn", "actor") in modelos:
    CARPETA_MODELOS.mkdir(exist_ok=True)
    for (familia, tarea), r in modelos.items():
        # Arquitectura (JSON) + pesos. Se guarda una copia SIN compilar: así no arrastra el estado de Adam,
        # que triplicaba el tamaño (610 MB en la primera prueba; sin él, ~210 MB). Es la misma forma de carga que usa 12.2.
        arquitectura = r["modelo"].to_json()
        limpio = keras.models.model_from_json(arquitectura)
        limpio.set_weights(r["modelo"].get_weights())
        (CARPETA_MODELOS / f"{familia}_{tarea}.json").write_text(arquitectura)
        limpio.save_weights(CARPETA_MODELOS / f"{familia}_{tarea}.weights.h5")
    (CARPETA_MODELOS / "clases.json").write_text(json.dumps({f"{f}_{t}": r["clases"] for (f, t), r in modelos.items()}, ensure_ascii=False))
    with zipfile.ZipFile(ZIP_MODELOS, "w", zipfile.ZIP_STORED) as z:
        for f in sorted(CARPETA_MODELOS.iterdir()):
            z.write(f, f.name)
    print(f"Guardado {ZIP_MODELOS} · {ZIP_MODELOS.stat().st_size / 1e6:.0f} MB · huella del dataset {HUELLA}")
    files.download(str(ZIP_MODELOS))           # queda en tu computadora: no se sube al repo ni al Drive del curso
else:
    print("No hay redes entrenadas en esta sesión (o GUARDAR_Y_DESCARGAR = False): nada que guardar.")

In [ ]:
# 12.2 Cargar las redes guardadas, para predecir sin volver a entrenar (alcanza con haber corrido la celda 1.1)
import json, zipfile

CARPETA_MODELOS = Path("/content/modelos_g6")
if "modelos" in globals() and ("cnn", "actor") in modelos:
    print("Las redes ya están en memoria (se entrenaron o cargaron en esta sesión): no hace falta cargarlas.")
else:
    if not (CARPETA_MODELOS / "clases.json").exists():
        if not os.path.exists("/content/drive/MyDrive"):
            from google.colab import drive
            drive.mount("/content/drive", readonly=True)   # solo lectura: busca el zip en tu Mi unidad, no escribe nada
        zips = sorted(glob.glob("/content/modelos_g6*.zip") + glob.glob("/content/drive/MyDrive/modelos_g6*.zip"))
        if not zips:
            from google.colab import files
            print("Subí el archivo modelos_g6.zip que descargó la celda 12.1:")
            zips = [os.path.abspath(n) for n in files.upload() if n.endswith(".zip")]
        assert zips, "No se encontró modelos_g6.zip"
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall(CARPETA_MODELOS)
    clases_guardadas = json.loads((CARPETA_MODELOS / "clases.json").read_text())
    modelos = {}
    for clave, clases in clases_guardadas.items():
        familia, tarea = clave.split("_", 1)
        m = keras.models.model_from_json((CARPETA_MODELOS / f"{clave}.json").read_text())
        try:
            m.load_weights(CARPETA_MODELOS / f"{clave}.weights.h5")
        except Exception:
            # Los zips anteriores a la versión de 210 MB guardaban también el estado de Adam:
            # se arma un Adam vacío con la misma forma para que los pesos encajen, y se reintenta.
            m.compile(optimizer=keras.optimizers.Adam(), loss="categorical_crossentropy")
            m.optimizer.build(m.trainable_variables)
            m.load_weights(CARPETA_MODELOS / f"{clave}.weights.h5")
        modelos[(familia, tarea)] = {"nombre": "CNN propia" if familia == "cnn" else "ResNet50 (conv4+conv5)",
                                     "modelo": m, "clases": clases}
    print("Redes cargadas:", ", ".join(f"{f} · {t}" for f, t in modelos))

In [ ]:
# 12.3 Funciones para predecir sobre una foto nueva
from PIL import ImageOps

NOMBRES = {"azzaro": "Azzaro", "candela_vetrano": "Candela Vetrano", "colapinto": "Colapinto", "dario_barassi": "Darío Barassi",
           "dolores_fonzi": "Dolores Fonzi", "erica_rivas": "Erica Rivas", "eva_de_dominici": "Eva De Dominici",
           "georgina_barbarossa": "Georgina Barbarossa", "griselda_siciliani": "Griselda Siciliani", "guillermo_francella": "Francella",
           "javier_milei": "Milei", "julio_chavez": "Julio Chávez", "lali_esposito": "Lali Espósito", "momo": "Momo",
           "mora_bianchi": "Mora Bianchi", "oscar_martinez": "Oscar Martínez", "pablo_echarri": "Pablo Echarri",
           "ricardo_darin": "Darín", "scaloni": "Scaloni", "tini_stoessel": "Tini Stoessel", "alegria": "alegría"}
EXTENSIONES = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
PANELES = [("cnn", "actor", "CNN propia · actor"), ("cnn", "emocion", "CNN propia · emoción"),
           ("resnet", "actor", "ResNet50 · actor"), ("resnet", "emocion", "ResNet50 · emoción")]


def preparar_foto(ruta, lado):
    # Igual que en entrenamiento: RGB en [0, 255] y del tamaño de cada red (la escala va dentro del modelo).
    # Corrige la orientación de las fotos de celular y recorta el cuadrado central, como los recortes del dataset.
    img = ImageOps.exif_transpose(Image.open(ruta)).convert("RGB")
    w, h = img.size; s = min(w, h)
    img = img.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s))
    return np.asarray(img.resize((lado, lado), Image.BILINEAR), dtype="float32")


def predecir(ruta, familia, tarea, top=5):
    # Las `top` clases más probables: [(nombre, probabilidad), ...].
    r = modelos[(familia, tarea)]
    prob = r["modelo"].predict(preparar_foto(ruta, r["modelo"].inputs[0].shape[1])[None], verbose=0)[0]
    return [(NOMBRES.get(r["clases"][i], r["clases"][i]), float(prob[i])) for i in np.argsort(-prob)[:top]]


def frase(pred, tarea, n=3):
    # "con un 67 % de confianza es Francella, con un 30 % es Darín y con un 3 % es Colapinto."
    verbo = "es" if tarea == "actor" else "expresa"
    partes = [f"con un {100 * pred[0][1]:.0f} % de confianza {verbo} {pred[0][0]}"]
    partes += [f"con un {100 * p:.0f} % " + ("es " if tarea == "actor" else "") + nombre for nombre, p in pred[1:n]]
    return ", ".join(partes[:-1]) + " y " + partes[-1] + "."


def mostrar_prediccion(ruta, top=5):
    preds = {(f, t): predecir(ruta, f, t, top) for f, t, _ in PANELES}
    fig, axes = plt.subplots(1, 5, figsize=(18, 3.8), gridspec_kw={"width_ratios": [1.1, 1, 1, 1, 1]})
    axes[0].imshow(preparar_foto(ruta, IMG_SIZE).astype("uint8")); axes[0].axis("off")
    axes[0].set_title(Path(ruta).name, fontsize=9)
    for a, (f, t, titulo) in zip(axes[1:], PANELES):
        nombres = [n for n, _ in preds[(f, t)]][::-1]; vals = [100 * p for _, p in preds[(f, t)]][::-1]
        a.barh(nombres, vals, color="#2E86AB" if f == "cnn" else "#E07A2E")
        for y, v in enumerate(vals):
            a.text(v + 1, y, f"{v:.0f} %", va="center", fontsize=8)
        a.set_xlim(0, 115); a.set_title(titulo, fontsize=10); a.tick_params(labelsize=8); a.set_xlabel("confianza (%)", fontsize=8)
    plt.tight_layout(); plt.show()
    for f, etiqueta in (("cnn", "CNN propia (desde cero)"), ("resnet", "ResNet50 (transfer learning)")):
        print(f"{etiqueta}\n   Actor: {frase(preds[(f, 'actor')], 'actor')}\n   Emoción: {frase(preds[(f, 'emocion')], 'emocion')}")
    print()

In [ ]:
# 12.4 Predecir todas las fotos de /content/fotos_nuevas y mostrar un resumen al final
from google.colab import files
assert "modelos" in globals() and ("cnn", "actor") in modelos, "Faltan las redes: correr todo el notebook o la celda 12.2."

CARPETA_FOTOS = Path("/content/fotos_nuevas")
CARPETA_FOTOS.mkdir(exist_ok=True)
rutas = sorted(str(p) for p in CARPETA_FOTOS.rglob("*") if p.is_file() and p.suffix.lower() in EXTENSIONES)
if not rutas:                                          # carpeta vacía: pide subir las fotos y las guarda ahí
    print("No hay fotos en /content/fotos_nuevas: elegí una o más para subir.")
    for nombre, contenido in files.upload().items():
        (CARPETA_FOTOS / nombre).write_bytes(contenido)
    rutas = sorted(str(p) for p in CARPETA_FOTOS.rglob("*") if p.is_file() and p.suffix.lower() in EXTENSIONES)
print(f"{len(rutas)} foto(s) en {CARPETA_FOTOS}\n")

filas = []
for ruta in rutas:
    mostrar_prediccion(ruta)                           # imagen, 4 gráficos de barras y las frases de cada red
    fila = {"foto": Path(ruta).name}
    for familia, red in (("cnn", "CNN"), ("resnet", "ResNet50")):
        for tarea, etiqueta in (("actor", "actor"), ("emocion", "emoción")):
            nombre, p = predecir(ruta, familia, tarea, top=1)[0]
            fila[f"{red} · {etiqueta}"] = f"{nombre} ({100 * p:.0f} %)"
    filas.append(fila)

if filas:
    print("Resumen: la opción más probable de cada red para cada foto")
    display(pd.DataFrame(filas).set_index("foto"))

### 12.5 Exportar las redes para la aplicación (ONNX)
La aplicación separa el modelo de los datos: el cliente sube una foto y solo ve la respuesta. Para eso las cuatro redes se exportan a **ONNX**, un formato estándar que corre igual en Python (`onnxruntime`, en la app de Streamlit del equipo) y en el navegador (`onnxruntime-web`, en el frontend de Lovable), sin TensorFlow ni servidor pago (D-024).

Antes de descargar, la celda **verifica** que cada archivo ONNX dé las mismas probabilidades que la red de Keras. Requiere las redes en memoria: «Ejecutar todas», o bien 1.1 → 12.2 → 12.3 → 12.5.

In [ ]:
# 12.5 Exportar las cuatro redes a ONNX, verificar contra Keras y descargar modelos_onnx.zip
import subprocess, sys, json, zipfile
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime"], check=True)
# tf2onnx sin dependencias: si no, baja protobuf a una versión vieja y rompe TensorFlow.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "tf2onnx"], check=True)
import onnxruntime as ort, tf2onnx
from keras.src.export.tf2onnx_lib import patch_tf2onnx
patch_tf2onnx()                                # el mismo arreglo que usa Keras: tf2onnx todavía no soporta numpy 2
from google.colab import files

assert "modelos" in globals() and ("cnn", "actor") in modelos, "Faltan las redes: correr 12.2 (o todo el notebook)."
assert "NOMBRES" in globals(), "Falta correr la celda 12.3 (nombres para mostrar)."
CARPETA_ONNX = Path("/content/modelos_onnx"); CARPETA_ONNX.mkdir(exist_ok=True)
ZIP_ONNX = Path("/content/modelos_onnx.zip")
rng = np.random.default_rng(SEED)
fotos = sorted(str(p) for p in Path("/content/fotos_nuevas").glob("*") if p.suffix.lower() in EXTENSIONES)

manifiesto = {"version": "E19", "huella_dataset": globals().get("HUELLA", "—"),
              "nota": "Entrada: RGB en [0, 255], NHWC, recorte cuadrado de la cara redimensionado al lado de cada red. "
                      "La escala va dentro de cada red. Salida: probabilidades (softmax).",
              "redes": {}}
for (familia, tarea), r in modelos.items():
    modelo, clave = r["modelo"], f"{familia}_{tarea}"
    lado = int(modelo.inputs[0].shape[1])
    ruta = CARPETA_ONNX / f"{clave}.onnx"
    # La red se llama con training=False: así la aumentación (capas Random*) no entra al grafo y las
    # BatchNorm usan sus promedios. (model.export la trazaba en modo entrenamiento y ONNX no soporta esas capas.)
    firma = [tf.TensorSpec((None, lado, lado, 3), tf.float32, name="imagen")]
    inferir = tf.function(lambda x, m=modelo: m(x, training=False), input_signature=firma)
    tf2onnx.convert.from_function(inferir, input_signature=firma, opset=17, output_path=str(ruta))
    sesion = ort.InferenceSession(str(ruta), providers=["CPUExecutionProvider"])
    entrada = sesion.get_inputs()[0].name

    # Verificación: imágenes al azar + las fotos de fotos_nuevas (si hay), Keras contra ONNX
    x = rng.uniform(0, 255, (4, lado, lado, 3)).astype("float32")
    if fotos:
        x = np.concatenate([x, np.stack([preparar_foto(f, lado) for f in fotos])])
    p_keras = modelo.predict(x, verbose=0)
    p_onnx = sesion.run(None, {entrada: x})[0]
    diferencia = float(np.abs(p_keras - p_onnx).max())
    assert diferencia < 1e-3, f"{clave}: ONNX no coincide con Keras (diferencia {diferencia:.2e})"

    manifiesto["redes"][clave] = {
        "archivo": ruta.name, "familia": familia, "tarea": tarea, "nombre": r["nombre"],
        "compite": familia == "cnn", "entrada": entrada, "lado": lado,
        "clases": r["clases"], "etiquetas": [NOMBRES.get(c, c) for c in r["clases"]]}
    print(f"{clave}: {ruta.stat().st_size / 1e6:.0f} MB · entrada '{entrada}' {lado}×{lado} · "
          f"diferencia máxima con Keras {diferencia:.1e} ✔")

(CARPETA_ONNX / "modelos.json").write_text(json.dumps(manifiesto, ensure_ascii=False, indent=1))
with zipfile.ZipFile(ZIP_ONNX, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(CARPETA_ONNX.iterdir()):
        z.write(f, f.name)
print(f"\nListo: {ZIP_ONNX} · {ZIP_ONNX.stat().st_size / 1e6:.0f} MB")
files.download(str(ZIP_ONNX))                  # a tu computadora; desde ahí se sube a Hugging Face (no a GitHub)

## 13. Conclusiones y limitaciones
Corrida final de este notebook (**E19**, partición sin fugas, D-022). Accuracy / F1 macro. Las corridas anteriores (E1–E17, partición vieja) están en el Anexo A y en `docs/EXPERIMENTOS_CNN.md`.

| Modelo | Identidad val | Emoción val | Identidad test | Emoción test |
|---|---|---|---|---|
| Trivial (clase más frecuente) | 0,145 / 0,013 | 0,241 / 0,078 | — | — |
| Random Forest (baseline) | 0,260 / 0,222 | 0,322 / 0,225 | — | — |
| **CNN propia · E19 (desde cero)** | **0,391 / 0,368** | **0,391 / 0,351** | **0,311 / 0,269** | **0,333 / 0,303** |
| ResNet50 conv4+conv5 (transfer learning) | 0,664 / 0,543 | 0,524 / 0,496 | 0,658 / 0,528 | 0,399 / 0,382 |

1. **El dato manda más que la arquitectura.** Con 20 actores de 2 a 20 videos cada uno (después de unificar), una CNN desde cero tiende a memorizar el video. Partir por video y controlar las fugas por contenido es lo que hace honestos estos números; corregir las fugas bajó la identidad de todos los modelos, incluido el Random Forest.
2. **La CNN propia supera a los baselines:** en validación, 0,391 contra 0,260 del Random Forest en identidad (+13 puntos) y 0,391 contra 0,322 en emoción (+7). En test, 0,311 en identidad (unas 6 veces el azar entre 20 actores) y 0,333 en emoción.
3. **El criterio de selección importó tanto como la arquitectura:** elegir por accuracy y no por pérdida (con pocos datos la red se vuelve sobreconfiada y `val_loss` deja de reflejar los aciertos) y separar las tareas en una red cada una fueron los dos saltos más grandes.
4. **La transferencia compensa la falta de datos:** ResNet50 más que duplica a la CNN en identidad (0,658 contra 0,311 en test). En emoción la ventaja es menor (0,399 contra 0,333); el acuerdo entre etiquetadores del TP1 (42,6 %) es una referencia aproximada de lo alcanzable.
5. **Los dos modelos confunden actores parecidos:** la CNN manda todas las fotos de test de Darín a Dolores Fonzi y las de Erica Rivas a Lali; ResNet50 lo sufre menos, pero también manda a Oscar Martínez a Francella y a Darín a Echarri. Eva De Dominici y Erica Rivas no las reconoce ninguno de los dos en validación. No se excluyó a ningún actor.
6. **La CNN sí usa la estructura espacial:** con los píxeles permutados cae de 0,391 a 0,241, mientras el MLP rinde igual (0,218 → 0,221).
7. **Fuera del dataset generaliza poco:** con 4 fotos de la web la CNN acertó 1 y ResNet50 ninguna como primera opción, a veces con 96–100 % de confianza. Aprendieron cómo se ve cada actor en esos videos (luz, peinado, cámara), no en cualquier foto.
8. **Limitaciones:** pocos videos por actor; en 9 actores validación y test comparten un video (R4); 435 imágenes de validación dan un número optimista y ruidoso (validación > test); la CNN de emoción varía entre corridas; el test se miró en más de una corrida a lo largo del trabajo (cada corrida completa lo evalúa al final) y por eso reportamos todas.

**Declaración de uso de IA generativa:** se usó un asistente de IA (Claude) para auditar el dataset, proponer la estructura del código y redactar borradores de la documentación. Todas las decisiones fueron revisadas por el grupo.

## Anexo A · Versiones anteriores de la CNN propia
Registro de todo el camino, con sus números en **validación** (accuracy / F1 macro). No se ejecuta: el código de cada versión está en `notebooks/TP2_Grupo6.ipynb` (secciones 7.1, 7.3 y Anexo A) y cada experimento, con hipótesis y lectura, en `docs/EXPERIMENTOS_CNN.md`.

### A.1 v1 — el experimento de control (NB11)
4 bloques `[Conv 3×3 → Conv 3×3 → BatchNorm → MaxPool]` con filtros 32 → 64 → 128 → 256, cierre `GAP → Dense(128) → Dropout(0,5) → softmax`, 224×224, una red por tarea, L2 1e-4, Adam 1e-3, early stopping. Aprende el video: ~89 % en train y ~29 % en validación. Identidad **0,287 / 0,249**, emoción **0,264 / 0,256**. Con píxeles permutados cae de 0,287 a 0,173 (sí usa la estructura espacial).

### A.2 v2 — residual con técnicas fuera del programa (descartada por no ser de la cátedra, D-015)
Red residual multitarea a 112×112 con alineación de rostros (detector YuNet), mixup, label smoothing, AdamW con tasa coseno, borrado y gris aleatorio, TTA y conjunto de redes. Funcionaba (mejor: E12), pero cada decisión tiene que poder defenderse con el material del curso.

| id | Configuración | Identidad | Emoción |
|---|---|---|---|
| E2 | rostro alineado 0,85, 120 ép. | 0,292 / 0,257 | 0,334 / 0,305 |
| E3 | recorte original 112×112, 120 ép. | 0,357 / 0,311 | 0,304 / 0,282 |
| E4–E7 | alineado 0,85 / 0,65 / 0,55, con y sin mixup, 80 ép. | 0,215–0,250 | 0,222–0,299 |
| E8–E9 | recorte original + mixup, 80 / 120 ép. | 0,311–0,334 | 0,271–0,343 |
| E10 | + zoom 0,2 / traslación 0,1 | 0,334 / 0,291 | 0,320 / 0,297 |
| E11 | recorte original a 64×64 | 0,343 / 0,319 | 0,299 / 0,288 |
| E12 | recorte original + mixup, 200 ép. | 0,360 / 0,344 | 0,418 / 0,400 |
| E13 | conjunto de 3 redes | 0,350 / 0,311 | 0,388 / 0,368 |

Lectura: alinear la cara **empeoró** (el recorte original conserva pelo y contorno, que ayudan a la identidad).

### A.3 v3 — solo técnicas de la cátedra (la línea que llegó a la versión final)
| id | Configuración | Identidad | Emoción |
|---|---|---|---|
| E14 | residual, una red con dos salidas; KerasTuner eligiendo por `val_loss` | 0,252 / 0,224 | 0,290 / 0,270 |
| E15 | KerasTuner eligiendo por **accuracy** de validación (D-017) | 0,355 / 0,281 | 0,350 / 0,305 |
| E16 · dos salidas | búsqueda afinada (D-018) | 0,355 / 0,284 | 0,362 / 0,302 |
| **E16 · una red por tarea** | **versión final (sección 7)** | **0,414 / 0,361** | **0,393 / 0,349** |

Lección de E14: el intento con menor `val_loss` acertaba el 6,5 % de los actores y uno con pérdida apenas mayor, el 35 %. Con pocos videos la red se vuelve sobreconfiada: la pérdida deja de reflejar los aciertos, por eso se elige por accuracy (como `objective="val_accuracy"` en NB09).